# Amazon ML Challenge 2026 — Track 02: Entity & Text Normalization Pipeline
**Objective:** Define and validate normalization choices for business names, addresses, and country codes.
- 1. Multilingual & Unicode accent handling (stripping diacritics while preserving non-Latin scripts like Devanagari)
- 2. Legal entity form standardization (LLC, Inc, Pvt Ltd, SARL, SAS, SCI)
- 3. Address component standardization (Street -> st, Road -> rd, PO Box, Suite/Unit)
- 4. Country code canonicalization (US, IN, FR)
- 5. Before vs. After comparison on verified ground truth matching pairs


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import src

print("Normalization module loaded from src.normalization")


## 1. Normalization Choices Explained
1. **Unicode NFKD Accent Stripping:** Normalizes `Léarning` -> `learning`, `Àmicale` -> `amicale`, while preserving Devanagari words like `राम`.
2. **Legal Entity Standardization:** Variations like `Pvt. Ltd.`, `Private Limited`, `L.L.C.`, `Inc.` are standardized or normalized to improve token matchability.
3. **Address Term Standardization:** Standardizes `Drive` -> `dr`, `Street` -> `st`, `Post Office Box` -> `pobox`.
4. **Punctuation & Case Folding:** Removes noisy leading symbols (`<<`, `--`, `+`) and lowercases all tokens.


In [ ]:
# Test individual normalization functions on sample edge cases
test_cases = [
    ("<< Team Ecole", "SARL Moncada Léarning Center", "Orelee's Barbershop"),
    ("राम मार्केटिंग प्राइवेट लिमिटेड", "Shri Supreme Consulting Private  (Limited)", "B+ Retail Inc"),
    ("1795 Westchester Drive, Unit G", "1 Ivanhoe Ave, PO Box 6009", "18 RUE JEN ZAY, Apt 4B")
]

print("=== BUSINESS NAME NORMALIZATION ===")
for name in test_cases[0] + test_cases[1]:
    norm = src.normalize_business_name(name)
    print(f"RAW:  '{name}'\nNORM: '{norm}'\n")

print("=== ADDRESS NORMALIZATION ===")
for addr in test_cases[2]:
    norm = src.normalize_address(addr)
    print(f"RAW:  '{addr}'\nNORM: '{norm}'\n")


## 2. Before vs. After Evaluation on Ground Truth Pairs
We load actual matched pairs from the ground truth and compute the increase in token overlap similarity.


In [ ]:
# Load sample ground truth and sources
s1 = src.load_source("train", 1, nrows=50000)
s2 = src.load_source("train", 2, nrows=100000)
s3 = src.load_source("train", 3, nrows=100000)
gt = src.load_ground_truth(nrows=20000)

s1_dict = s1.set_index("entity_id").to_dict("index")
target_dict = {**s2.set_index("entity_id").to_dict("index"), **s3.set_index("entity_id").to_dict("index")}
gt_pairs = src.parse_ground_truth_pairs(gt)

comparisons = []
for _, row in gt_pairs.iterrows():
    s1_id = row["source1_entity_id"]
    cand_id = row["matched_entity_id"]
    if s1_id in s1_dict and cand_id in target_dict:
        s1_raw = s1_dict[s1_id]["business_name"]
        cand_raw = target_dict[cand_id]["business_name"]
        
        s1_norm = src.normalize_business_name(s1_raw)
        cand_norm = src.normalize_business_name(cand_raw)
        
        raw_jaccard = src.token_jaccard_similarity(set(str(s1_raw).lower().split()), set(str(cand_raw).lower().split()))
        norm_jaccard = src.token_jaccard_similarity(set(s1_norm.split()), set(cand_norm.split()))
        
        comparisons.append({
            "s1_raw": s1_raw,
            "target_raw": cand_raw,
            "raw_jaccard": round(raw_jaccard, 3),
            "norm_jaccard": round(norm_jaccard, 3),
            "improvement": round(norm_jaccard - raw_jaccard, 3)
        })
        if len(comparisons) >= 15:
            break

comp_df = pd.DataFrame(comparisons)
print("=== TOKEN OVERLAP BEFORE VS AFTER NORMALIZATION ===")
display(comp_df)


In [ ]:
# Batch vector normalization demonstration
print("Testing batch normalization performance on S1 (50,000 records)...")
norm_s1 = src.normalize_dataframe(s1)
print("Batch normalization completed successfully! New columns added:")
print(norm_s1[["entity_id", "norm_name", "norm_address", "norm_country"]].head(5))
